# Schema-Miner with OpenRouter

This tutorial demonstrates how to use **Schema-Miner** with **OpenRouter** for remote large language model inference.

We use:

- **Schema-Miner** for scientific schema mining and iterative schema refinement
- **OpenRouter** as the model API service
- **`qwen/qwen3-235b-a22b`** as the example model
- The OpenRouter OpenAI-compatible API endpoint for remote inference

Because inference is performed remotely through the OpenRouter API, **no local GPU is required**.

This notebook demonstrates the complete three-stage Schema-Miner workflow:

1. **Stage 1 — Initial Schema Mining**
2. **Stage 2 — Preliminary Schema Refinement**
3. **Stage 3 — Final Schema Refinement**

Stages 2 and 3 can be run iteratively over one or more batches of scientific papers, incorporating expert feedback between successive runs.

> Schema-Miner accesses OpenRouter through its OpenAI-compatible API backend.

In [1]:
%pip install -U schema-miner

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import sys
import sysconfig
import importlib.metadata
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Schema-Miner:", importlib.metadata.version("schema-miner"))

scripts_dir = Path(sysconfig.get_path("scripts"))
cli_name = "schema-miner.exe" if os.name == "nt" else "schema-miner"
SCHEMA_MINER_CLI = scripts_dir / cli_name

print("Schema-Miner CLI:", SCHEMA_MINER_CLI)
print("CLI available:", SCHEMA_MINER_CLI.exists())

Python: 3.12.10
Schema-Miner: 3.2.5
Schema-Miner CLI: c:\Users\DSouzaJ\AppData\Local\Programs\Python\Python312\Scripts\schema-miner.exe
CLI available: True


## 2. Stage 1 — Initial Schema Mining

Stage 1 generates an initial schema from a process specification document using a large language model accessed through **OpenRouter**.

This tutorial uses:

- **Service:** OpenRouter
- **Model:** `qwen/qwen3-235b-a22b`
- **API endpoint:** `https://openrouter.ai/api/v1`

The OpenRouter API key is entered securely at runtime and is not written to the notebook or stored in a `.env` file.

> Schema-Miner accesses OpenRouter through its OpenAI-compatible `SAIA` backend. Therefore, the notebook configures the OpenRouter API key and base URL through Schema-Miner's `SAIA_API_KEY` and `SAIA_BASE_URL` environment variables.

This tutorial assumes the following structure:

```text
data/
└── stage1/
    ├── process.txt
    ├── process-description.pdf
    ├── feedback/
    └── schema/
```

The `process.txt` file contains the process name and process description:

```text
process name: <process name>
process description: <process description>
```

The process specification is provided as a single PDF in `data/stage1/`.

The `schema/` directory stores the schema generated by Stage 1. The `feedback/` directory is used later to provide expert feedback for Stage 2.

In [7]:
# Stage 1 — Initial Schema Mining with OpenRouter

import os
import subprocess
from getpass import getpass
from pathlib import Path

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

STAGE1_DIR = Path("../../data/stage1").resolve()

DEFAULT_MODEL = "qwen/qwen3-235b-a22b"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

model = input(
    f"OpenRouter model [{DEFAULT_MODEL}]: "
).strip() or DEFAULT_MODEL

api_key = getpass("OpenRouter API key: ")

if not api_key:
    raise ValueError("An OpenRouter API key is required.")

# ------------------------------------------------------------------
# Read process information
# ------------------------------------------------------------------

process_txt = STAGE1_DIR / "process.txt"

if not process_txt.exists():
    raise FileNotFoundError(
        f"Missing process file: {process_txt}"
    )

text = process_txt.read_text(encoding="utf-8").strip()
lines = text.splitlines()

process_name = None
description_lines = []
in_description = False

for line in lines:
    stripped = line.strip()
    lower = stripped.lower()

    if lower.startswith("process name:"):
        process_name = stripped.split(":", 1)[1].strip()
        in_description = False

    elif lower.startswith("process description:"):
        description_lines.append(
            stripped.split(":", 1)[1].strip()
        )
        in_description = True

    elif in_description:
        description_lines.append(stripped)

process_description = "\n".join(description_lines).strip()

if not process_name:
    raise ValueError(
        "Missing 'process name:' in process.txt"
    )

if not process_description:
    raise ValueError(
        "Missing 'process description:' in process.txt"
    )

# ------------------------------------------------------------------
# Locate the process specification
# ------------------------------------------------------------------

pdfs = sorted(STAGE1_DIR.glob("*.pdf"))

if len(pdfs) != 1:
    raise ValueError(
        f"Expected exactly one PDF in {STAGE1_DIR}, "
        f"found {len(pdfs)}."
    )

specification_pdf = pdfs[0]

# ------------------------------------------------------------------
# Output directory
# ------------------------------------------------------------------

results_dir = STAGE1_DIR / "schema"
results_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Runtime environment for Schema-Miner
# ------------------------------------------------------------------

env = os.environ.copy()

env.update({
    # Schema-Miner uses its SAIA backend for OpenAI-compatible endpoints.
    "LLM_PROVIDER": "SAIA",
    "LLM_MODEL": model,
    "SAIA_API_KEY": api_key,
    "SAIA_BASE_URL": OPENROUTER_BASE_URL,
    "PROCESS_NAME": process_name,
    "PROCESS_DESCRIPTION": process_description,
    "STAGE1_SPECS_PATH": str(specification_pdf),
    "STAGE2_PAPERS_PATH": "",
    "STAGE3_PAPERS_PATH": "",
    "RESULTS_PATH": str(results_dir),
})

# Remove the separate Python variable.
# The API key remains only in the environment passed to Schema-Miner.
del api_key

# ------------------------------------------------------------------
# Run Stage 1
# ------------------------------------------------------------------

command = [
    str(SCHEMA_MINER_CLI),
    "--stage", "1",
]

log_path = results_dir / "stage1.log"

print(f"Process:             {process_name}")
print(f"Model:               {model}")
print("Service:             OpenRouter")
print(f"OpenRouter endpoint: {OPENROUTER_BASE_URL}")
print(f"Specification:       {specification_pdf}")
print(f"Results:             {results_dir}")
print()

try:
    process = subprocess.Popen(
        command,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    output_lines = []

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)

    return_code = process.wait()

    # Save the complete Schema-Miner output.
    log_path.write_text(
        "".join(output_lines),
        encoding="utf-8",
    )

    if return_code == 0:

        # ----------------------------------------------------------
        # Normalize the OpenRouter model filename
        # ----------------------------------------------------------

        # Schema-Miner may replace "/" in the model identifier with "-",
        # producing, for example:
        #
        # qwen-qwen3-235b-a22b.json
        #
        # For the tutorial workflow we use only the model basename:
        #
        # qwen3-235b-a22b.json

        generated_model_file = model.replace("/", "-")
        normalized_model_file = model.split("/")[-1]

        generated_schema = (
            results_dir / f"{generated_model_file}.json"
        )

        normalized_schema = (
            results_dir / f"{normalized_model_file}.json"
        )

        if (
            generated_schema.exists()
            and generated_schema != normalized_schema
        ):
            if normalized_schema.exists():
                normalized_schema.unlink()

            generated_schema.rename(normalized_schema)

            print(
                f"\nSchema filename normalized to: "
                f"{normalized_schema.name}"
            )

        print("\nStage 1 completed successfully.")
        print(f"Results: {results_dir}")

    else:
        print("\n" + "=" * 70)
        print("Stage 1 did not complete successfully.")
        print(
            f"Schema-Miner exited with status code {return_code}."
        )
        print(f"Full log: {log_path}")
        print(
            "Review the Schema-Miner output above for the "
            "underlying error, correct the issue, and rerun this cell."
        )
        print("=" * 70)

except FileNotFoundError:
    print(
        "\nThe Schema-Miner CLI could not be found. "
        "Run the installation and environment verification cells first."
    )

except KeyboardInterrupt:
    if "process" in locals() and process.poll() is None:
        process.terminate()
        process.wait()

    print("\nStage 1 was interrupted by the user.")

Process:             Metal-organic cages synthesis
Model:               qwen/qwen3-235b-a22b
Service:             OpenRouter
OpenRouter endpoint: https://openrouter.ai/api/v1
Specification:       C:\Users\DSouzaJ\Code\schema-miner\data\stage1\process-description.pdf
Results:             C:\Users\DSouzaJ\Code\schema-miner\data\stage1\schema

Running SCHEMA-MINER -- Stage 1: Initial Schema Extraction
2026-08-11 15:47:38,674 - LLMs4SchemaDiscovery Framework -- A Human-in-the-Loop Workflow for Scientific Schema Mining with Large Language Models for Metal-organic cages synthesis process
2026-08-11 15:47:38,674 - Stage 1: Initial Schema Mining
2026-08-11 15:47:38,674 - Reading the process specification document...
2026-08-11 15:47:38,674 - Extracting text from the PDF: process-description.pdf
2026-08-11 15:47:38,847 - PDF parsed successfully
2026-08-11 15:47:38,847 - Performing LLM (qwen/qwen3-235b-a22b) Inference to extract schema...
2026-08-11 15:47:40,541 - Using SAIA - LLM Inference with

## 3. Stage 2 — Preliminary Schema Refinement

Stage 2 refines the initial schema using scientific papers together with expert feedback from the preceding schema-mining run.

The papers may be organized into **one or more batches**. A batch may contain a single paper or several papers, depending on how frequently expert review should be incorporated.

For example:

```text
data/
└── stage2/
    ├── batch1/
    │   ├── paper-1.pdf
    │   ├── paper-2.pdf
    │   └── ...
    ├── batch2/
    │   ├── paper-6.pdf
    │   ├── paper-7.pdf
    │   └── ...
    ├── feedback-batch1/
    ├── feedback-batch2/
    ├── schema-batch1/
    └── schema-batch2/
```

The refinement proceeds iteratively:

```text
Stage 1 schema
      +
Stage 1 expert feedback
      +
Stage 2 / batch1 papers
      ↓
Stage 2 / schema-batch1
      ↓
expert review
      ↓
Stage 2 / feedback-batch1
      +
Stage 2 / batch2 papers
      ↓
Stage 2 / schema-batch2
```

After each batch, inspect the resulting schema and place the corresponding expert feedback in the appropriate `feedback-batchN/` directory before running the next batch.

The number of batches is not fixed. Papers may be divided into two or more batches, or processed one paper at a time. The important requirement is that **each new batch uses the schema and expert feedback resulting from review of the previous run**.

Run the following cell once for each Stage 2 batch. Available batches are detected automatically from `data/stage2/`.

In [9]:
# Stage 2 — Preliminary Schema Refinement with OpenRouter

import subprocess
from pathlib import Path

STAGE1_DIR = Path("../../data/stage1").resolve()
STAGE2_DIR = Path("../../data/stage2").resolve()

# ------------------------------------------------------------------
# Detect available batches
# ------------------------------------------------------------------

available_batches = sorted(
    int(path.name.replace("batch", ""))
    for path in STAGE2_DIR.glob("batch*")
    if path.is_dir() and path.name.replace("batch", "").isdigit()
)

if not available_batches:
    raise FileNotFoundError(
        f"No batch directories found in {STAGE2_DIR}"
    )

print("Available Stage 2 batches:", available_batches)

batch = int(
    input(
        f"Stage 2 batch to run {available_batches}: "
    ).strip()
)

if batch not in available_batches:
    raise ValueError(
        f"Batch {batch} not found. Available batches: {available_batches}"
    )

# ------------------------------------------------------------------
# Resolve model-specific filename
# ------------------------------------------------------------------

# OpenRouter model identifiers may contain a provider prefix such as:
# qwen/qwen3-235b-a22b
#
# Schema and feedback files use only the model basename:
# qwen3-235b-a22b

model_file = model.split("/")[-1]

# ------------------------------------------------------------------
# Resolve schema and expert feedback from the preceding run
# ------------------------------------------------------------------

if batch == 1:
    # First Stage 2 batch starts from Stage 1.

    schema_path = (
        STAGE1_DIR
        / "schema"
        / f"{model_file}.json"
    )

    feedback_path = (
        STAGE1_DIR
        / "feedback"
        / f"{model_file}.txt"
    )

else:
    # Later batches continue from the preceding Stage 2 batch.

    previous_batch = batch - 1

    schema_path = (
        STAGE2_DIR
        / f"schema-batch{previous_batch}"
        / f"{model_file}.json"
    )

    feedback_path = (
        STAGE2_DIR
        / f"feedback-batch{previous_batch}"
        / f"{model_file}.txt"
    )

# ------------------------------------------------------------------
# Current papers and output directory
# ------------------------------------------------------------------

papers_dir = STAGE2_DIR / f"batch{batch}"

results_dir = STAGE2_DIR / f"schema-batch{batch}"
results_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Validate inputs
# ------------------------------------------------------------------

if not schema_path.exists():
    raise FileNotFoundError(
        f"Schema from the preceding run not found: {schema_path}"
    )

if not feedback_path.exists():
    raise FileNotFoundError(
        f"Expert feedback from the preceding run not found: {feedback_path}"
    )

pdfs = sorted(papers_dir.glob("*.pdf"))

if not pdfs:
    raise FileNotFoundError(
        f"No PDF papers found in {papers_dir}"
    )

# ------------------------------------------------------------------
# Runtime environment for Schema-Miner
# ------------------------------------------------------------------

# Reuse the authenticated OpenRouter environment established in Stage 1.
# Schema-Miner accesses OpenRouter through its SAIA/OpenAI-compatible backend.

env = dict(env)

env.update({
    "LLM_PROVIDER": "SAIA",
    "LLM_MODEL": model,
    "PROCESS_NAME": process_name,
    "PROCESS_DESCRIPTION": process_description,
    "STAGE1_SPECS_PATH": "",
    "STAGE2_PAPERS_PATH": str(papers_dir),
    "STAGE3_PAPERS_PATH": "",
    "RESULTS_PATH": str(results_dir),
})

# ------------------------------------------------------------------
# Run Stage 2
# ------------------------------------------------------------------

command = [
    str(SCHEMA_MINER_CLI),
    "--stage", "2",
    "--schema", str(schema_path),
    "--expert-feedback", str(feedback_path),
    "--papers", "all",
]

log_path = results_dir / f"stage2-batch{batch}.log"

print(f"Running Stage 2 — Batch {batch}")
print(f"Model:           {model}")
print("Service:         OpenRouter")
print(f"Input schema:    {schema_path}")
print(f"Expert feedback: {feedback_path}")
print(f"Papers:          {papers_dir} ({len(pdfs)} PDFs)")
print(f"Results:         {results_dir}")
print()

try:
    process = subprocess.Popen(
        command,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    output_lines = []

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)

    return_code = process.wait()

    # Save the complete Schema-Miner output.
    log_path.write_text(
        "".join(output_lines),
        encoding="utf-8",
    )

    if return_code == 0:

        # ----------------------------------------------------------
        # Normalize the generated schema filename
        # ----------------------------------------------------------

        generated_model_file = model.replace("/", "-")
        normalized_model_file = model.split("/")[-1]

        generated_schema = (
            results_dir / f"{generated_model_file}.json"
        )

        normalized_schema = (
            results_dir / f"{normalized_model_file}.json"
        )

        if (
            generated_schema.exists()
            and generated_schema != normalized_schema
        ):
            if normalized_schema.exists():
                normalized_schema.unlink()

            generated_schema.rename(normalized_schema)

            print(
                f"\nSchema filename normalized to: "
                f"{normalized_schema.name}"
            )

        print("\nStage 2 completed successfully.")
        print(f"Results: {results_dir}")

    else:
        print("\n" + "=" * 70)
        print(
            f"Stage 2 — Batch {batch} did not complete successfully."
        )
        print(
            f"Schema-Miner exited with status code {return_code}."
        )
        print(f"Full log: {log_path}")
        print(
            "Review the Schema-Miner output above for the underlying "
            "error, correct the issue, and rerun this cell."
        )
        print("=" * 70)

except FileNotFoundError:
    print(
        "\nThe Schema-Miner CLI could not be found. "
        "Run the installation and environment verification cells first."
    )

except KeyboardInterrupt:
    if "process" in locals() and process.poll() is None:
        process.terminate()
        process.wait()

    print("\nStage 2 was interrupted by the user.")

Available Stage 2 batches: [1, 2]
Running Stage 2 — Batch 2
Model:           qwen/qwen3-235b-a22b
Service:         OpenRouter
Input schema:    C:\Users\DSouzaJ\Code\schema-miner\data\stage2\schema-batch1\qwen3-235b-a22b.json
Expert feedback: C:\Users\DSouzaJ\Code\schema-miner\data\stage2\feedback-batch1\qwen3-235b-a22b.txt
Papers:          C:\Users\DSouzaJ\Code\schema-miner\data\stage2\batch2 (2 PDFs)
Results:         C:\Users\DSouzaJ\Code\schema-miner\data\stage2\schema-batch2

Running SCHEMA-MINER -- Stage 2: Preliminary Schema Refinement
Total papers: 2 | Batch size: 2 | Total batches: 1

Batch 1/1:
Processing paper 1/2: C:\Users\DSouzaJ\Code\schema-miner\data\stage2\batch2\Zhang_2017.pdf
2026-08-11 15:55:33,776 - LLMs4SchemaDiscovery Framework -- A Human-in-the-Loop Workflow for Scientific Schema Mining with Large Language Models for Metal-organic cages synthesis process
2026-08-11 15:55:33,776 - Stage 2: Preliminary Schema Refinement
2026-08-11 15:55:33,776 - Reading the schema...

## 4. Stage 3 — Final Schema Refinement

Stage 3 performs the final refinement of the schema using scientific papers together with expert feedback from the preceding schema-mining run.

As in Stage 2, the papers may be organized into **one or more batches**. A batch may contain a single paper or several papers, depending on how frequently expert review should be incorporated.

For example:

```text
data/
└── stage3/
    ├── batch1/
    │   ├── paper-1.pdf
    │   ├── paper-2.pdf
    │   └── ...
    ├── batch2/
    │   ├── paper-6.pdf
    │   ├── paper-7.pdf
    │   └── ...
    ├── feedback-batch1/
    ├── schema-batch1/
    └── schema-batch2/
```

Stage 3 begins from the **latest completed Stage 2 schema together with the expert feedback created after reviewing that schema**.

The refinement then proceeds iteratively:

```text
Latest Stage 2 schema
      +
Latest Stage 2 expert feedback
      +
Stage 3 / batch1 papers
      ↓
Stage 3 / schema-batch1
      ↓
expert review
      ↓
Stage 3 / feedback-batch1
      +
Stage 3 / batch2 papers
      ↓
Stage 3 / schema-batch2
```

For subsequent Stage 3 batches, each run uses the schema and expert feedback resulting from review of the preceding Stage 3 batch.

The number of batches is not fixed. Papers may be divided into two or more batches, or processed one paper at a time. The important requirement is that **each new batch uses the schema and expert feedback resulting from review of the previous run**.

Expert feedback only needs to be created when another refinement batch will follow. If the current batch is the final Stage 3 batch, no additional feedback directory is required.

Run the following cell once for each Stage 3 batch. Available batches are detected automatically from `data/stage3/`.

In [11]:
# Stage 3 — Final Schema Refinement with OpenRouter

import subprocess
from pathlib import Path

STAGE2_DIR = Path("../../data/stage2").resolve()
STAGE3_DIR = Path("../../data/stage3").resolve()

# ------------------------------------------------------------------
# Detect available Stage 3 batches
# ------------------------------------------------------------------

available_batches = sorted(
    int(path.name.replace("batch", ""))
    for path in STAGE3_DIR.glob("batch*")
    if path.is_dir() and path.name.replace("batch", "").isdigit()
)

if not available_batches:
    raise FileNotFoundError(
        f"No batch directories found in {STAGE3_DIR}"
    )

print("Available Stage 3 batches:", available_batches)

batch = int(
    input(
        f"Stage 3 batch to run {available_batches}: "
    ).strip()
)

if batch not in available_batches:
    raise ValueError(
        f"Batch {batch} not found. Available batches: {available_batches}"
    )

# ------------------------------------------------------------------
# Resolve model-specific filename
# ------------------------------------------------------------------

# OpenRouter model identifiers may contain a provider prefix such as:
# qwen/qwen3-235b-a22b
#
# Schema and feedback files use only the model basename:
# qwen3-235b-a22b

model_file = model.split("/")[-1]

# ------------------------------------------------------------------
# Resolve schema and expert feedback from the preceding run
# ------------------------------------------------------------------

if batch == 1:
    # Stage 3 starts from the latest completed Stage 2 checkpoint
    # for which both schema and corresponding expert feedback exist.

    stage2_checkpoints = []

    for schema_dir in STAGE2_DIR.glob("schema-batch*"):
        suffix = schema_dir.name.replace("schema-batch", "")

        if not suffix.isdigit():
            continue

        stage2_batch = int(suffix)

        schema_candidate = (
            schema_dir
            / f"{model_file}.json"
        )

        feedback_candidate = (
            STAGE2_DIR
            / f"feedback-batch{stage2_batch}"
            / f"{model_file}.txt"
        )

        if schema_candidate.exists() and feedback_candidate.exists():
            stage2_checkpoints.append(
                (
                    stage2_batch,
                    schema_candidate,
                    feedback_candidate,
                )
            )

    if not stage2_checkpoints:
        raise FileNotFoundError(
            "No completed Stage 2 checkpoint with both schema "
            "and expert feedback was found."
        )

    (
        source_stage2_batch,
        schema_path,
        feedback_path,
    ) = max(
        stage2_checkpoints,
        key=lambda item: item[0]
    )

else:
    # Later Stage 3 batches continue from the preceding Stage 3 batch.

    previous_batch = batch - 1

    schema_path = (
        STAGE3_DIR
        / f"schema-batch{previous_batch}"
        / f"{model_file}.json"
    )

    feedback_path = (
        STAGE3_DIR
        / f"feedback-batch{previous_batch}"
        / f"{model_file}.txt"
    )

# ------------------------------------------------------------------
# Current papers and output directory
# ------------------------------------------------------------------

papers_dir = STAGE3_DIR / f"batch{batch}"

results_dir = STAGE3_DIR / f"schema-batch{batch}"
results_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Validate inputs
# ------------------------------------------------------------------

if not schema_path.exists():
    raise FileNotFoundError(
        f"Schema from the preceding run not found: {schema_path}"
    )

if not feedback_path.exists():
    raise FileNotFoundError(
        f"Expert feedback from the preceding run not found: {feedback_path}"
    )

pdfs = sorted(papers_dir.glob("*.pdf"))

if not pdfs:
    raise FileNotFoundError(
        f"No PDF papers found in {papers_dir}"
    )

# ------------------------------------------------------------------
# Runtime environment for Schema-Miner
# ------------------------------------------------------------------

# Reuse the authenticated OpenRouter environment established in Stage 1.
# Schema-Miner accesses OpenRouter through its SAIA/OpenAI-compatible backend.

env = dict(env)

env.update({
    "LLM_PROVIDER": "SAIA",
    "LLM_MODEL": model,
    "PROCESS_NAME": process_name,
    "PROCESS_DESCRIPTION": process_description,
    "STAGE1_SPECS_PATH": "",
    "STAGE2_PAPERS_PATH": "",
    "STAGE3_PAPERS_PATH": str(papers_dir),
    "RESULTS_PATH": str(results_dir),
})

# ------------------------------------------------------------------
# Run Stage 3
# ------------------------------------------------------------------

command = [
    str(SCHEMA_MINER_CLI),
    "--stage", "3",
    "--schema", str(schema_path),
    "--expert-feedback", str(feedback_path),
    "--papers", "all",
]

log_path = results_dir / f"stage3-batch{batch}.log"

print(f"Running Stage 3 — Batch {batch}")
print(f"Model:           {model}")
print("Service:         OpenRouter")

if batch == 1:
    print(f"Starting from:   Stage 2 — Batch {source_stage2_batch}")

print(f"Input schema:    {schema_path}")
print(f"Expert feedback: {feedback_path}")
print(f"Papers:          {papers_dir} ({len(pdfs)} PDFs)")
print(f"Results:         {results_dir}")
print()

try:
    process = subprocess.Popen(
        command,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    output_lines = []

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)

    return_code = process.wait()

    # Save the complete Schema-Miner output.
    log_path.write_text(
        "".join(output_lines),
        encoding="utf-8",
    )

    if return_code == 0:

        # ----------------------------------------------------------
        # Normalize the generated schema filename
        # ----------------------------------------------------------

        generated_model_file = model.replace("/", "-")
        normalized_model_file = model.split("/")[-1]

        generated_schema = (
            results_dir / f"{generated_model_file}.json"
        )

        normalized_schema = (
            results_dir / f"{normalized_model_file}.json"
        )

        if (
            generated_schema.exists()
            and generated_schema != normalized_schema
        ):
            if normalized_schema.exists():
                normalized_schema.unlink()

            generated_schema.rename(normalized_schema)

            print(
                f"\nSchema filename normalized to: "
                f"{normalized_schema.name}"
            )

        print("\nStage 3 completed successfully.")
        print(f"Results: {results_dir}")

    else:
        print("\n" + "=" * 70)
        print(
            f"Stage 3 — Batch {batch} did not complete successfully."
        )
        print(
            f"Schema-Miner exited with status code {return_code}."
        )
        print(f"Full log: {log_path}")
        print(
            "Review the Schema-Miner output above for the underlying "
            "error, correct the issue, and rerun this cell."
        )
        print("=" * 70)

except FileNotFoundError:
    print(
        "\nThe Schema-Miner CLI could not be found. "
        "Run the installation and environment verification cells first."
    )

except KeyboardInterrupt:
    if "process" in locals() and process.poll() is None:
        process.terminate()
        process.wait()

    print("\nStage 3 was interrupted by the user.")

Available Stage 3 batches: [1, 2]
Running Stage 3 — Batch 2
Model:           qwen/qwen3-235b-a22b
Service:         OpenRouter
Input schema:    C:\Users\DSouzaJ\Code\schema-miner\data\stage3\schema-batch1\qwen3-235b-a22b.json
Expert feedback: C:\Users\DSouzaJ\Code\schema-miner\data\stage3\feedback-batch1\qwen3-235b-a22b.txt
Papers:          C:\Users\DSouzaJ\Code\schema-miner\data\stage3\batch2 (3 PDFs)
Results:         C:\Users\DSouzaJ\Code\schema-miner\data\stage3\schema-batch2

Running SCHEMA-MINER -- Stage 3: Finalize Schema Refinement
Total papers: 3 | Batch size: 3 | Total batches: 1

Batch 1/1:
Processing paper 1/3: C:\Users\DSouzaJ\Code\schema-miner\data\stage3\batch2\Chemistry A European J - 2024 - Ward - Chameleonic Cages  Encapsulation of Anionic  Neutral  and Cationic Guest Species.pdf
2026-08-11 16:10:18,919 - LLMs4SchemaDiscovery Framework -- A Human-in-the-Loop Workflow for Scientific Schema Mining with Large Language Models for Metal-organic cages synthesis process
2026-0